In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [13]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [14]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/diabetes.csv")

print("Dataset Shape:", df.shape)

Dataset Shape: (768, 9)


In [15]:
cols_to_replace = ['Glucose','BloodPressure','SkinThickness','Insulin','BMI']

for col in cols_to_replace:
    df[col] = df[col].replace(0, df[col].median())

df.dropna(inplace=True)

In [16]:
X = df.drop("Outcome", axis=1)
y = df["Outcome"]

In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [18]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [19]:
model = LogisticRegression()
model.fit(X_train_scaled, y_train)

LogisticRegression()

In [20]:
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

In [22]:
print("\n========== MODEL EVALUATION ==========")
print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


========== MODEL EVALUATION ==========
Accuracy: 0.745

Confusion Matrix:
 [[124  27]
 [ 32  48]]

Classification Report:
               precision    recall  f1-score   support

           0       0.79      0.82      0.81       151
           1       0.64      0.60      0.62        80

    accuracy                           0.74       231
   macro avg       0.72      0.71      0.71       231
weighted avg       0.74      0.74      0.74       231



In [24]:
risk_levels = []

for prob in y_prob:
    if prob < 0.4:

        risk_levels.append("Low Risk")
    elif prob < 0.7:
        risk_levels.append("Medium Risk")
    else:
        risk_levels.append("High Risk")

risk_df = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted_Probability": y_prob,
    "Risk_Level": risk_levels
})

print("\n========== SAMPLE RISK PROFILING ==========")
print(risk_df.head())


========== SAMPLE RISK PROFILING ==========
   Actual  Predicted_Probability   Risk_Level
0       0               0.231695     Low Risk
1       0               0.192190     Low Risk
2       0               0.115358     Low Risk
3       0               0.130295     Low Risk
4       0               0.496332  Medium Risk


In [26]:
print("\n========== NEW PATIENT PREDICTION ==========")

new_patient = pd.DataFrame(
    [[6, 148, 72, 35, 100, 33.6, 0.627, 50]],
    columns=X.columns
)

new_scaled = scaler.transform(new_patient)

prediction = model.predict(new_scaled)
probability = model.predict_proba(new_scaled)[0][1]

if probability < 0.4:
    risk = "Low Risk"

elif probability < 0.7:
    risk = "Medium Risk"
else:
    risk = "High Risk"

print("Prediction:", "Diabetic" if prediction[0]==1 else "Non-Diabetic")
print("Risk Level:", risk)
print("Probability:", round(probability, 3))

print("\nProgram Completed Successfully")


========== NEW PATIENT PREDICTION ==========
Prediction: Diabetic
Risk Level: High Risk
Probability: 0.742

Program Completed Successfully
